# Data Import

In [1]:
import pandas as pd

In [6]:
pfad = r"C:\Users\annic\OneDrive\Dokumente\Master Technischer Umweltschutz\Master Thesis\Emission_Trajectory_Thesis\Data\321-140-17133-26-Bestand.csv"

stock = pd.read_csv(pfad, encoding="latin1", sep=";")
stock

,Berichtsjahr,Segment,Antriebsart,Jahr der Erstzulassung,Anzahl
0,01.01.2017,Minis,Benzin,1990,954
1,01.01.2017,Minis,Benzin,1991,1774
2,01.01.2017,Minis,Benzin,1992,2396
3,01.01.2017,Minis,Benzin,1993,3957
4,01.01.2017,Minis,Benzin,1994,5480
...,...,...,...,...,...
26429,01.01.2026,Sonstige,Erdgas (CNG) (einschl. bivalent),2012,20
26430,01.01.2026,Sonstige,Erdgas (CNG) (einschl. bivalent),2013,22
26431,01.01.2026,Sonstige,Erdgas (CNG) (einschl. bivalent),2014,10
26432,01.01.2026,Sonstige,Erdgas (CNG) (einschl. bivalent),2017,2


In [7]:
pfad = r"C:\Users\annic\OneDrive\Dokumente\Master Technischer Umweltschutz\Master Thesis\Emission_Trajectory_Thesis\Data\321-140-17133-26-ABS.csv"

exits = pd.read_csv(pfad, encoding="latin1", sep=";")
exits

,Berichtsjahr,Segment,Antriebsart,Jahr der Erstzulassung,Anzahl
0,2016,Minis,Benzin,1990,202
1,2016,Minis,Benzin,1991,391
2,2016,Minis,Benzin,1992,728
3,2016,Minis,Benzin,1993,1850
4,2016,Minis,Benzin,1994,2626
...,...,...,...,...,...
21214,2025,Sonstige,Erdgas (CNG) (einschl. bivalent),2003,3
21215,2025,Sonstige,Erdgas (CNG) (einschl. bivalent),2007,1
21216,2025,Sonstige,Erdgas (CNG) (einschl. bivalent),2012,9
21217,2025,Sonstige,Erdgas (CNG) (einschl. bivalent),2013,11


In [8]:
pfad = r"C:\Users\annic\OneDrive\Dokumente\Master Technischer Umweltschutz\Master Thesis\Emission_Trajectory_Thesis\Data\321-140-17133-26-NZL.csv"

entries = pd.read_csv(pfad, encoding="latin1", sep=";")
entries

,Berichtsjahr,Segment,Antriebsart,Anzahl
0,2016,Minis,Benzin,230152
1,2016,Minis,Diesel,3946
2,2016,Minis,Elektro (BEV),954
3,2016,Minis,Hybrid,1
4,2016,Minis,Flüssiggas (LPG) (einschl. bivalent),749
...,...,...,...,...
784,2025,Sonstige,Diesel,399
785,2025,Sonstige,Elektro (BEV),4258
786,2025,Sonstige,Hybrid,2816
787,2025,Sonstige,Sonstige,30


# Creation of dictionaries

In [20]:
import pandas as pd

file_path = r"C:\Users\annic\OneDrive\Dokumente\Master Technischer Umweltschutz\Master Thesis\Emission_Trajectory_Thesis\Data\321-140-17133-26-Bestand.csv"

stock = pd.read_csv(file_path, encoding="latin1", sep=";")

# Filter stock data for reporting year 2026
stock_2026_df = stock[stock["Berichtsjahr"] == "01.01.2026"].copy()

# Get all segments and powertrains available in the 2026 data
segments = stock_2026_df["Segment"].unique()
powertrains = stock_2026_df["Antriebsart"].unique()

# Create dictionary:
# stock_2026_dict[segment][powertrain] -> DataFrame
stock_2026_dict = {}

for segment in segments:
    stock_2026_dict[segment] = {}

    for powertrain in powertrains:
        df = stock_2026_df[
            (stock_2026_df["Segment"] == segment) &
            (stock_2026_df["Antriebsart"] == powertrain)
        ].copy()

        df = df.rename(
            columns={
                "Jahr der Erstzulassung": "Year of first registration",
                "Anzahl": "Number of vehicles",
            }
        )

        df = df[["Year of first registration", "Number of vehicles"]]

        df["Year of first registration"] = pd.to_numeric(
            df["Year of first registration"],
            errors="coerce"
        )

        df["Number of vehicles"] = pd.to_numeric(
            df["Number of vehicles"],
            errors="coerce"
        ).fillna(0)

        df = df.dropna(subset=["Year of first registration"])

        df["Year of first registration"] = df["Year of first registration"].astype(int)
        df["Number of vehicles"] = df["Number of vehicles"].astype(int)

        df = (
            df.groupby("Year of first registration", as_index=False)["Number of vehicles"]
            .sum()
            .sort_values("Year of first registration", ascending=False)
            .reset_index(drop=True)
        )

        stock_2026_dict[segment][powertrain] = df

In [21]:
stock_2026_dict["Geländewagen"]["Benzin"]

,Year of first registration,Number of vehicles
0,2025,25215
1,2024,40428
2,2023,63287
3,2022,43775
4,2021,48584
5,2020,68861
6,2019,101212
7,2018,92108
8,2017,73442
9,2016,50017


In [22]:
stock_2026_dict["Sonstige"]["Erdgas (CNG) (einschl. bivalent)"]

,Year of first registration,Number of vehicles
0,2018,1
1,2017,2
2,2014,10
3,2013,22
4,2012,20
5,2009,1
6,2007,1
7,2005,1
8,2004,1
9,2003,3
